In [4]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import shap
from torch.nn import Module

# 定义与原始训练脚本相同的模型类结构
class MyModel(nn.Module):
    def __init__(self, input_dim=12):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 450)
        self.bn1 = nn.BatchNorm1d(450)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(450, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# ====================== 1. 定义路径 ======================
model_dir = r"D:\文成数据库\训练模型3.4-KQ"
shap_output_dir = r"D:\文成数据库\4.20KQ-SHAP分析"

# 创建SHAP输出目录
if not os.path.exists(shap_output_dir):
    os.makedirs(shap_output_dir)

# ====================== 2. 加载模型和数据 ======================
# 加载原始数据
df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库3.4-成分-性能.xlsx")

# 加载模型
model_path = os.path.join(model_dir, 'kq_model.pth')
model_structure_path = os.path.join(model_dir, 'KQ_model.joblib')
scaler_path = os.path.join(model_dir, 'final_scaler.joblib')
features_path = os.path.join(model_dir, 'final_features.txt')

# 读取特征列表
with open(features_path, 'r', encoding='utf-8') as f:
    final_features = [line.strip() for line in f.readlines()]

print("加载模型使用的特征:", final_features)

# 加载标准化器
scaler = joblib.load(scaler_path)

# 加载模型（两种方式尝试）
try:
    # 方式1：直接加载joblib保存的模型
    model_structure = joblib.load(model_structure_path)
    model = model_structure
    print("成功通过joblib直接加载模型")
except Exception as e:
    print(f"通过joblib加载模型失败，错误: {e}")
    print("尝试通过加载模型参数的方式重建模型...")
    # 方式2：创建新模型实例并加载参数
    model = MyModel(len(final_features))

try:
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    print("成功加载模型权重")
except Exception as e:
    print(f"加载模型权重出错: {e}")
    raise

device = torch.device("cpu")  # 使用CPU可能更稳定
model.to(device)
model.eval()

# ====================== 3. 为SHAP准备数据 ======================
# 提取特征数据，保存原始特征值
X_original = df[final_features].values  # 保存原始值
X_scaled = scaler.transform(X_original)  # 标准化数据
y = df['KQ'].values

# 转换为PyTorch张量
X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

# ====================== 4. 创建自定义模型包装器 =====================
class ShapModelWrapper:
    def __init__(self, model):
        self.model = model
        
    def __call__(self, X):
        if isinstance(X, np.ndarray):
            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        else:
            X_tensor = X
            
        with torch.no_grad():
            output = self.model(X_tensor)
            
        return output.cpu().numpy()

# ====================== 5. SHAP分析 - 使用Kernel解释器 ======================
print("创建SHAP解释器...")

# 参数说明: 
# - max_bg_samples: 背景数据集的大小，增加可提高准确性但会增加计算时间
# - 背景数据集用于为SHAP值提供基准，代表"平均"或"预期"的模型输出
max_bg_samples = min(100, X_scaled.shape[0])  # 可以调整背景样本数量
bg_indices = np.random.choice(X_scaled.shape[0], max_bg_samples, replace=False)
bg_data = X_scaled[bg_indices]

# 参数说明:
# - 可以选择使用所有数据点或随机选择部分样本进行SHAP分析
# - 样本数量越多，结果越准确但计算时间也越长
print(f"使用所有 {X_scaled.shape[0]} 个样本进行SHAP分析")
sample_data = X_scaled
sample_original = X_original  # 保存对应的原始数据
sample_indices = np.arange(X_scaled.shape[0])

# 使用KernelExplainer而非DeepExplainer
# 参数说明:
# - KernelExplainer适用于任何模型，但计算较慢
# - 第一个参数是模型函数，第二个参数是背景数据
model_wrapper = ShapModelWrapper(model)
explainer = shap.KernelExplainer(model_wrapper, bg_data)
print("成功创建SHAP KernelExplainer")

print("计算SHAP值...")
# 参数说明:
# - 可以在这里设置nsamples参数来控制近似精度，值越大越精确但计算时间也越长
# - 例如: shap_values = explainer.shap_values(sample_data, nsamples=100)
shap_values = explainer.shap_values(sample_data)

# 检查SHAP值形状
print(f"原始SHAP值形状: {np.array(shap_values).shape}")

# 如果是列表，取第一个元素（单输出模型）
if isinstance(shap_values, list) and len(shap_values) == 1:
    shap_values = shap_values[0]

# 确保SHAP值是2D数组
if len(np.array(shap_values).shape) > 2:
    shap_values = np.squeeze(shap_values)
    
# 打印最终形状
print(f"处理后SHAP值形状: {np.array(shap_values).shape}")

# ====================== 6. 保存SHAP分析结果 ======================
feature_names_list = [str(f) for f in final_features]

# ====================== 6.1 计算并保存平均绝对SHAP值表格（特征重要性） ======================
print("计算特征重要性...")
try:
    # 计算每个特征的平均绝对SHAP值
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    
    # 创建DataFrame
    shap_importance_df = pd.DataFrame({
        'Feature': feature_names_list,
        'Mean_Absolute_SHAP_Value': mean_abs_shap,
        'Relative_Importance': mean_abs_shap / np.sum(mean_abs_shap) * 100  # 相对重要性百分比
    })
    
    # 按平均绝对SHAP值排序
    shap_importance_df = shap_importance_df.sort_values('Mean_Absolute_SHAP_Value', ascending=False)
    shap_importance_df.to_excel(os.path.join(shap_output_dir, 'shap_feature_importance.xlsx'), index=False)
    print("成功创建特征重要性Excel文件")
    
except Exception as e:
    print(f"创建特征重要性表格时出错: {e}")

# ====================== 6.2 创建并保存每个样本的原始特征值、SHAP值和预测值表格 ======================
print("保存原始特征值和SHAP值...")
try:
    # 获取所有样本的预测值
    with torch.no_grad():
        all_predictions = model(torch.tensor(sample_data, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    print(f"成功计算出 {len(all_predictions)} 个样本的预测值")
    
    # 创建包含原始特征值的DataFrame (使用未标准化的原始值)
    feature_values_df = pd.DataFrame(sample_original, columns=feature_names_list)
    print(f"成功创建原始特征值DataFrame，形状: {feature_values_df.shape}")
    
    # 创建包含SHAP值的DataFrame
    shap_values_df = pd.DataFrame(shap_values, columns=[f'SHAP_{col}' for col in feature_names_list])
    print(f"成功创建SHAP值DataFrame，形状: {shap_values_df.shape}")
    
    # 获取基准值 - 确保它是一个标量值
    base_value = explainer.expected_value
    if hasattr(base_value, "__len__") and len(base_value) == 1:
        base_value = base_value[0]
    print(f"基准值: {base_value}")
    
    # 创建一个与样本数量长度相同的基准值数组
    base_values = np.full(len(sample_indices), base_value)
    
    # 创建包含样本索引和预测值的DataFrame
    predictions_df = pd.DataFrame({
        'sample_index': sample_indices,
        'prediction': all_predictions,
        'base_value': base_values  # 使用长度匹配的数组
    })
    print(f"成功创建预测值DataFrame，形状: {predictions_df.shape}")
    
    # 合并所有DataFrame
    combined_df = pd.concat([predictions_df, feature_values_df, shap_values_df], axis=1)
    print(f"成功合并所有DataFrame，最终形状: {combined_df.shape}")
    
    # 保存文件
    file_path = os.path.join(shap_output_dir, 'features_shap_predictions.xlsx')
    combined_df.to_excel(file_path, index=False)
    
    # 验证文件是否已创建
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # 转换为MB
        print(f"成功创建原始特征值、SHAP值和预测值Excel文件: {file_path}")
        print(f"文件大小: {file_size:.2f} MB")
    else:
        print(f"警告: 文件 {file_path} 可能未成功创建!")
    
except Exception as e:
    print(f"创建原始特征值和SHAP值表格时出错: {e}")
    import traceback
    traceback.print_exc()  # 打印详细错误堆栈

# ====================== 6.3 计算并保存10个样本的力图数据 ======================
print("计算力图数据...")

# 创建力图数据专用目录
force_dir = os.path.join(shap_output_dir, 'force_data')
if not os.path.exists(force_dir):
    os.makedirs(force_dir)

# 为力图创建专用的数据结构
force_data = []
force_samples = min(10, len(sample_data))  # 控制生成的样本数量

for i in range(force_samples):
    try:
        # 收集力图数据
        base_value = explainer.expected_value
        # 获取预测值
        with torch.no_grad():
            prediction = model(torch.tensor(sample_data[i:i+1], dtype=torch.float32).to(device)).item()
        
        # 将所有数据保存到字典中
        sample_force_data = {
            'sample_index': sample_indices[i],
            'base_value': base_value,
            'prediction': prediction,
            'difference': prediction - base_value
        }
        
        # 为每个特征添加原始值和SHAP值
        for j, feature in enumerate(feature_names_list):
            sample_force_data[f'feature_{j+1}_name'] = feature
            sample_force_data[f'feature_{j+1}_value'] = sample_original[i, j]  # 使用原始值
            sample_force_data[f'feature_{j+1}_shap'] = shap_values[i, j]
        
        force_data.append(sample_force_data)
        print(f"成功计算样本 {sample_indices[i]} 的力图数据")
        
    except Exception as e:
        print(f"处理样本 {i} 的力图数据时出错: {e}")

# 将力图数据保存到Excel
try:
    if force_data:
        # 创建水平格式的汇总Excel
        force_df_horizontal = pd.DataFrame(force_data)
        force_df_horizontal.to_excel(os.path.join(force_dir, 'force_all_samples_summary.xlsx'), index=False)
        
        # 为每个样本创建单独的Excel文件
        for sample_dict in force_data:
            sample_idx = sample_dict['sample_index']
            base_value = sample_dict['base_value']
            prediction = sample_dict['prediction']
            
            # 提取特征数据，创建垂直格式
            features = []
            values = []
            shap_values_list = []
            
            # 获取所有特征及其SHAP值
            feature_count = 0
            for key in sample_dict:
                if key.startswith('feature_') and key.endswith('_name'):
                    idx = key.split('_')[1]
                    feature_name = sample_dict[f'feature_{idx}_name']
                    feature_value = sample_dict[f'feature_{idx}_value']
                    feature_shap = sample_dict[f'feature_{idx}_shap']
                    
                    features.append(feature_name)
                    values.append(feature_value)
                    shap_values_list.append(feature_shap)
                    feature_count += 1
            
            # 按SHAP值绝对值排序（从大到小）
            sorted_indices = np.argsort(-np.abs(shap_values_list))
            sorted_features = [features[i] for i in sorted_indices]
            sorted_values = [values[i] for i in sorted_indices]
            sorted_shap_values = [shap_values_list[i] for i in sorted_indices]
            
            # 计算起始位置和结束位置（用于绘制力图）
            cumulative = [base_value]
            for shap_val in sorted_shap_values:
                cumulative.append(cumulative[-1] + shap_val)
            
            start_positions = cumulative[:-1]  # 不包括最后一个值
            end_positions = cumulative[1:]     # 不包括第一个值
            
            # 创建更适合Origin绘制力图的格式
            origin_force_df = pd.DataFrame({
                'x_position': range(len(sorted_features) + 1),  # 包括基准值
                'feature': ['BASE VALUE'] + sorted_features,
                'start_value': start_positions,
                'end_value': end_positions,
                'shap_value': [0] + sorted_shap_values,
                'is_positive': [None] + list(np.array(sorted_shap_values) > 0),
                'feature_value': [None] + sorted_values,
                'feature_rank': [0] + list(range(1, feature_count + 1)),
                'bar_width': 0.8,  # 力图绘制时的柱宽
                'bar_height': np.array(end_positions) - np.array(start_positions)
            })
            
            # 添加基本信息
            origin_force_df['sample_index'] = sample_idx
            origin_force_df['base_value'] = base_value
            origin_force_df['prediction'] = prediction
            
            # 保存到单独的Excel文件
            sample_file = os.path.join(force_dir, f'force_sample_{sample_idx}.xlsx')
            
            with pd.ExcelWriter(sample_file) as writer:
                # 1. 主要格式 - 适合力图绘制
                origin_force_df.to_excel(writer, sheet_name='力图数据', index=False)
                
                # 2. 简化版 - 只包含绘图所需的核心列
                core_columns = ['x_position', 'feature', 'start_value', 'end_value', 'shap_value', 'is_positive']
                origin_force_df[core_columns].to_excel(writer, sheet_name='简化绘图数据', index=False)
                
                # 3. 只包含特征和SHAP值的简表
                feature_shap_df = pd.DataFrame({
                    'feature': sorted_features,
                    'shap_value': sorted_shap_values,
                    'is_positive': np.array(sorted_shap_values) > 0,
                    'feature_value': sorted_values
                })
                feature_shap_df.to_excel(writer, sheet_name='特征SHAP值', index=False)
            
            print(f"成功创建样本 {sample_idx} 的独立力图数据文件")
        
        print(f"成功保存力图数据到独立文件夹: {force_dir}")
    else:
        print("没有力图数据可以保存")
except Exception as e:
    print(f"保存力图数据时出错: {e}")

# ====================== 6.4 计算并保存热图数据 ======================
print("准备热图数据并保存到Excel...")
try:
    # 创建热图数据帧 - 使用所有样本
    heatmap_excel_data = pd.DataFrame()
    
    # 首先添加样本索引
    heatmap_excel_data['样本索引'] = sample_indices
    
    # 获取所有样本的预测值
    with torch.no_grad():
        all_predictions = model(torch.tensor(sample_data, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    
    # 获取基准值 (SHAP explainer的expected_value)
    # 确保基准值是标量，不是数组
    base_value = explainer.expected_value
    if hasattr(base_value, "__len__") and len(base_value) == 1:
        base_value = base_value[0]
    print(f"SHAP基准值: {base_value}")
    
    # 显式创建与样本数相同长度的基准值列
    base_values = np.full(len(sample_data), base_value)
    heatmap_excel_data['基准值'] = base_values
    
    # 添加预测值和与基准值的差异
    heatmap_excel_data['预测值'] = all_predictions
    heatmap_excel_data['预测值与基准值差异'] = all_predictions - base_values
    
    # 添加每个特征的SHAP值
    for i, feat in enumerate(feature_names_list):
        heatmap_excel_data[f"{feat}_SHAP"] = shap_values[:, i]
    
    # 添加特征值 (使用原始值)
    for i, feat in enumerate(feature_names_list):
        heatmap_excel_data[f"{feat}_值"] = sample_original[:, i]
    
    # 添加SHAP值之和（应接近于预测值与基准值的差异）
    heatmap_excel_data['SHAP值总和'] = np.sum(shap_values, axis=1)
    
    # 保存到Excel
    excel_path = os.path.join(shap_output_dir, 'heatmap_data.xlsx')
    heatmap_excel_data.to_excel(excel_path, index=False)
    print(f"热图数据已保存到Excel - 包含所有 {len(sample_data)} 个样本")
    print(f"文件路径: {excel_path}")

except Exception as e:
    print(f"保存热图数据时出错: {e}")
    print(f"错误详情: {str(e)}")
    import traceback
    traceback.print_exc()

print(f"SHAP分析完成! 结果已保存到: {shap_output_dir}")

# 输出特征重要性排名
try:
    print("\n特征重要性排名 (基于平均绝对SHAP值):")
    for i, (_, row) in enumerate(shap_importance_df.iterrows(), 1):
        print(f"{i}. {row['Feature']}: {row['Mean_Absolute_SHAP_Value']:.4f} ({row['Relative_Importance']:.2f}%)")
except Exception as e:
    print(f"打印特征重要性排名时出错: {e}")

加载模型使用的特征: ['Δx', 'ΔHmix', 'am', 'Ω', 'mean_C4 enthalpy melting', 'mean_Electrophilicity Index']
成功通过joblib直接加载模型
成功加载模型权重
创建SHAP解释器...
使用所有 221 个样本进行SHAP分析
成功创建SHAP KernelExplainer
计算SHAP值...


  0%|          | 0/221 [00:00<?, ?it/s]

原始SHAP值形状: (221, 6, 1)
处理后SHAP值形状: (221, 6)
计算特征重要性...
成功创建特征重要性Excel文件
保存原始特征值和SHAP值...
成功计算出 221 个样本的预测值
成功创建原始特征值DataFrame，形状: (221, 6)
成功创建SHAP值DataFrame，形状: (221, 6)
基准值: 11.959183435440064
成功创建预测值DataFrame，形状: (221, 3)
成功合并所有DataFrame，最终形状: (221, 15)
成功创建原始特征值、SHAP值和预测值Excel文件: D:\文成数据库\4.20KQ-SHAP分析\features_shap_predictions.xlsx
文件大小: 0.04 MB
计算力图数据...
成功计算样本 0 的力图数据
成功计算样本 1 的力图数据
成功计算样本 2 的力图数据
成功计算样本 3 的力图数据
成功计算样本 4 的力图数据
成功计算样本 5 的力图数据
成功计算样本 6 的力图数据
成功计算样本 7 的力图数据
成功计算样本 8 的力图数据
成功计算样本 9 的力图数据
保存力图数据时出错: Per-column arrays must each be 1-dimensional
准备热图数据并保存到Excel...
SHAP基准值: 11.959183435440064
热图数据已保存到Excel - 包含所有 221 个样本
文件路径: D:\文成数据库\4.20KQ-SHAP分析\heatmap_data.xlsx
SHAP分析完成! 结果已保存到: D:\文成数据库\4.20KQ-SHAP分析

特征重要性排名 (基于平均绝对SHAP值):
1. mean_C4 enthalpy melting: 3.3989 (29.22%)
2. am: 2.7215 (23.40%)
3. mean_Electrophilicity Index: 2.0412 (17.55%)
4. Δx: 1.8258 (15.70%)
5. Ω: 1.0551 (9.07%)
6. ΔHmix: 0.5900 (5.07%)
